In [ ]:
import fitz
import chromadb
from sentence_transformers import SentenceTransformer
import re, os
import requests

In [ ]:
pdf_path = "D:/bfsi_rag/rbi/kyc.pdf"
doc = fitz.open(pdf_path)
full_text = ""
for page_num, page in enumerate(doc):
    full_text += f"\n[PAGE {page_num+1}]\n{page.get_text()}"
doc.close()

print(f"Total characters: {len(full_text)}")
print(full_text[:500])  # preview first 500 chars

In [ ]:
def chunk_text(text, chunk_size=400, overlap=80):
    paragraphs = re.split(r'\n\s*\n', text)
    paragraphs = [p.strip() for p in paragraphs if p.strip()]
    chunks = []
    current_chunk = ""
    for para in paragraphs:
        if len(current_chunk) + len(para) + 2 <= chunk_size:
            current_chunk += ("\n\n" if current_chunk else "") + para
        else:
            if current_chunk:
                chunks.append(current_chunk)
            carry = current_chunk[-overlap:] if len(current_chunk) > overlap else current_chunk
            current_chunk = carry + ("\n\n" if carry else "") + para
    if current_chunk:
        chunks.append(current_chunk)
    return chunks

In [ ]:
chunks = chunk_text(full_text)

In [ ]:
print(f"Number of chunks: {len(chunks)}")
print(chunks[0])  # inspect first chunk
print("---"*20)
print(chunks[1])
print("---"*20)
print(chunks[2])
print("---"*20)
print(chunks[3])  # inspect another

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(chunks, show_progress_bar=True)
print(embeddings.shape)

In [ ]:
client = chromadb.PersistentClient(path="chroma_store")
try:
    client.delete_collection("bfsi_docs")
except:
    pass
collection = client.get_or_create_collection("bfsi_docs", metadata={"hnsw:space": "cosine"})

ids = [f"kyc_chunk_{i}" for i in range(len(chunks))]
metadatas = [{"source": "kyc_guidelines.pdf", "chunk_index": i} for i in range(len(chunks))]

collection.add(ids=ids, embeddings=embeddings.tolist(), documents=chunks, metadatas=metadatas)
print("Stored successfully")

In [ ]:
query = "What percentage of ownership defines a beneficial owner in a trust under this Master Direction?"
query_embedding = model.encode([query]).tolist()

results = collection.query(query_embeddings=query_embedding, n_results=3, 
                            include=["documents", "metadatas", "distances"])

for i in range(len(results["documents"][0])):
    print(f"Distance: {results['distances'][0][i]:.4f}")
    print(results["documents"][0][i][:200])
    print("---"*30)

In [ ]:
def clean_chunk(text):
    text = re.sub(r'\[PAGE \d+\]', '', text)
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)
    return text.strip()

In [ ]:
context = "\n\n---\n\n".join(clean_chunk(c) for c in results["documents"][0])
prompt = f"""You are a compliance assistant. Answer using ONLY this context.
If not found, say so.

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""

response = requests.post("http://localhost:11434/api/generate", 
                          json={"model": "phi3:mini", "prompt": prompt, "stream": False},
                          timeout=400)
print(response.json()["response"])

In [27]:
# ── Config (validated values) ────────────────────
CHROMA_PATH = "chroma_store"
COLLECTION_NAME = "bfsi_docs"
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
TOP_K = 3
OLLAMA_URL = "http://localhost:11434/api/generate"
LLM_MODEL = "phi3:mini"
CONFIDENCE_THRESHOLD = 0.55

In [28]:
def ask(query: str, top_k: int = TOP_K) -> dict:
    """
    Full RAG pipeline as a single callable function.
    Assumes `model` and `collection` already exist in notebook scope
    (from your earlier cells).
    """
    # Retrieve
    query_embedding = model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    chunks = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    best_distance = distances[0] if distances else 1.0
    low_confidence = best_distance > CONFIDENCE_THRESHOLD

    # Build prompt
    context = "\n\n---\n\n".join(chunks)
    prompt = f"""You are a compliance assistant for a financial institution.
Answer the question below using ONLY the provided context.
If the answer is not present in the context, say: "I could not find relevant information in the provided documents."

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""

    # Call Phi-3
    response = requests.post(
        OLLAMA_URL,
        json={"model": LLM_MODEL, "prompt": prompt, "stream": False},
        timeout=120
    )
    answer = response.json()["response"].strip()

    sources = [
        {"source": m["source"], "chunk_index": m["chunk_index"], "distance": round(d, 4)}
        for m, d in zip(metadatas, distances)
    ]

    return {
        "query": query,
        "answer": answer,
        "sources": sources,
        "best_distance": round(best_distance, 4),
        "low_confidence": low_confidence
    }

In [29]:
result = ask("What percentage of ownership defines a beneficial owner in a trust?")

print("ANSWER:", result["answer"])
print("\nSOURCES:")
for s in result["sources"]:
    print(f"  - chunk #{s['chunk_index']} | distance: {s['distance']}")
print(f"\nLow confidence: {result['low_confidence']}")

ANSWER: 10 percent or more interest in the trust.

SOURCES:
  - chunk #13 | distance: 0.416
  - chunk #89 | distance: 0.5111
  - chunk #198 | distance: 0.5852

Low confidence: False


In [6]:
from presidio_analyzer import PatternRecognizer, Pattern, AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider

# ── Custom Recognizer: PAN (Permanent Account Number) ──
# Format: 5 letters, 4 digits, 1 letter (e.g., ABCDE1234F)
pan_pattern = Pattern(
    name="pan_pattern",
    regex=r"\b[A-Z]{5}[0-9]{4}[A-Z]\b",
    score=0.85
)
pan_recognizer = PatternRecognizer(
    supported_entity="IN_PAN",
    patterns=[pan_pattern],
    context=["pan", "permanent account number", "income tax"]
)

# ── Custom Recognizer: Aadhaar ──
# Format: 12 digits, optionally grouped as 4-4-4 with spaces or hyphens
aadhaar_pattern = Pattern(
    name="aadhaar_pattern",
    regex=r"\b\d{4}[\s-]?\d{4}[\s-]?\d{4}\b",
    score=0.7
)
aadhaar_recognizer = PatternRecognizer(
    supported_entity="IN_AADHAAR",
    patterns=[aadhaar_pattern],
    context=["aadhaar", "aadhar", "uidai", "uid"]
)

# ── Custom Recognizer: Indian Mobile Number ──
# Format: 10 digits starting with 6-9, optional +91 / 0 prefix
indian_phone_pattern = Pattern(
    name="indian_phone_pattern",
    regex=r"(\+91[\-\s]?|0)?[6-9]\d{9}\b",
    score=0.6
)
indian_phone_recognizer = PatternRecognizer(
    supported_entity="IN_PHONE_NUMBER",
    patterns=[indian_phone_pattern],
    context=["phone", "mobile", "contact", "number"]
)

# ── Build analyzer and register custom recognizers ──
analyzer = AnalyzerEngine()
analyzer.registry.add_recognizer(pan_recognizer)
analyzer.registry.add_recognizer(aadhaar_recognizer)
analyzer.registry.add_recognizer(indian_phone_recognizer)

print("Custom recognizers registered:", ["IN_PAN", "IN_AADHAAR", "IN_PHONE_NUMBER"])

Custom recognizers registered: ['IN_PAN', 'IN_AADHAAR', 'IN_PHONE_NUMBER']


In [7]:
test_text = "My name is Rohan Sharma, my PAN is ABCDE1234F, Aadhaar is 1234 5678 9012, and my number is 9876543210."

results = analyzer.analyze(
    text=test_text,
    language="en",
    entities=["PERSON", "IN_PAN", "IN_AADHAAR", "IN_PHONE_NUMBER"]  # restrict to relevant entities
)

for r in results:
    print(f"Entity: {r.entity_type} | Score: {r.score:.2f} | Text: '{test_text[r.start:r.end]}'")

Entity: IN_PAN | Score: 1.00 | Text: 'ABCDE1234F'
Entity: IN_AADHAAR | Score: 1.00 | Text: '1234 5678 9012'
Entity: IN_PHONE_NUMBER | Score: 0.95 | Text: '9876543210'
Entity: PERSON | Score: 0.85 | Text: 'Rohan Sharma'
